# 5. Random sampling, warm start and CEM

This notebook extends `02_random_sampling_warm_start.ipynb` with the **Cross-Entropy Method (CEM)**.
The robot model, cost, obstacles, horizon, sample count and seed are the same as in notebook 02.

At every control step:
1. Shift the previous best plan forward by one control step (warm start).
2. Sample candidate plans and simulate them, exactly as in notebook 02.
3. Select the lowest-cost plans (**elites**) and estimate their mean and covariance.
4. Repeat sampling and fitting `cem_iterations` times, without moving the robot.
5. Apply only the first control of the best evaluated plan and repeat.

All batches use uniform sampling, as in notebook 02. Elite statistics adapt the perturbations.
The covariance is carried between control steps and can be reset every `covariance_reset_steps` steps.

**Requirements:** Python, NumPy, Matplotlib, and a Jupyter notebook environment.
If needed, run `%pip install numpy matplotlib` in a separate cell. Then run top to bottom.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Re-running this cell resets the random sequence.
rng = np.random.default_rng(7)
dt = 0.1                         # Control interval [s]
horizon = 20                     # Future controls: 2 seconds
num_samples = 512                # Candidate plans per CEM iteration
max_steps = 200                  # Maximum simulation length
goal_tolerance = 0.15            # Stop within this distance [m]
start = np.array([0.0, 0.0, 0.0]) # x [m], y [m], heading [rad]
goal = np.array([4.0, 2.0])       # Target position [m]
control_low = np.array([-1.0, -1.5])  # Minimum v [m/s], omega [rad/s]
control_high = np.array([1.0, 1.5])  # Maximum v [m/s], omega [rad/s]

# Each row contains obstacle center x [m], center y [m], and radius [m].
obstacles = np.array([
    [1.20, 0.35, 0.28],
    [2.15, 1.05, 0.35],
    [3.05, 1.35, 0.30],
])
robot_radius = 0.15  # Robot footprint [m]
safety_margin = 0.15  # Desired extra clearance [m]
obstacle_weight = 250.0
collision_weight = 10000.0
# CEM settings: these are the new tunable parameters compared with notebook 02.
num_elites = 32                  # Fit the distribution to the best candidates
cem_iterations = 3              # Optimization iterations per control step (>= 1)
covariance_reset_steps = 10      # Reset every X control steps; None disables resets
initial_noise = np.array([0.2, 0.4])  # Uniform half-widths, exactly as in notebook 02
min_std = 0.02                  # Regularization: prevents covariance collapse

assert 2 <= num_elites <= num_samples
assert isinstance(cem_iterations, int) and cem_iterations >= 1
assert covariance_reset_steps is None or (
    isinstance(covariance_reset_steps, int) and covariance_reset_steps >= 1)
assert np.all(initial_noise > 0) and min_std > 0

# Flatten a plan in the order [v_0, omega_0, v_1, omega_1, ...].
# A full covariance also learns correlations between different prediction steps.
# Uniform noise in [-a, a] has variance a ** 2 / 3.
initial_covariance = np.diag(np.tile(initial_noise ** 2 / 3.0, horizon))
covariance_floor = min_std ** 2 * np.eye(2 * horizon)


## 1. Predict the robot motion

We use forward Euler integration of the unicycle kinematics:

$$x_{t+1}=x_t+\Delta t\,v_t\cos\theta_t,\qquad
 y_{t+1}=y_t+\Delta t\,v_t\sin\theta_t,\qquad
 \theta_{t+1}=\theta_t+\Delta t\,\omega_t.$$

The same function handles one state with shape `(3,)` or a batch with shape `(num_samples, 3)`.
Batching candidates keeps the notebook fast while retaining an explicit loop over prediction time.


In [ ]:
def robot_step(state, control):
    """Advance one state, or a batch of states, by one time step."""
    next_state = np.empty_like(state)
    next_state[..., 0] = state[..., 0] + dt * control[..., 0] * np.cos(state[..., 2])
    next_state[..., 1] = state[..., 1] + dt * control[..., 0] * np.sin(state[..., 2])
    next_state[..., 2] = state[..., 2] + dt * control[..., 1]
    return next_state


def rollout(initial_state, controls):
    """Simulate controls shaped (candidates, horizon, 2)."""
    states = np.empty((len(controls), horizon + 1, 3))
    states[:, 0] = initial_state
    for t in range(horizon):
        states[:, t + 1] = robot_step(states[:, t], controls[:, t])
    return states


def trajectory_cost(states, controls):
    """Score goal progress, control effort, and obstacle proximity."""
    squared_distance = np.sum((states[:, 1:, :2] - goal) ** 2, axis=2)
    effort = controls[:, :, 0] ** 2 + 0.1 * controls[:, :, 1] ** 2
    # Surface-to-surface clearance includes the robot's circular footprint.
    obstacle_cost = np.zeros(len(controls))
    for obstacle_x, obstacle_y, radius in obstacles:
        distance = np.linalg.norm(
            states[:, 1:, :2] - np.array([obstacle_x, obstacle_y]), axis=2)
        clearance = distance - radius - robot_radius
        margin_violation = np.maximum(safety_margin - clearance, 0.0)
        obstacle_cost += np.sum(obstacle_weight * margin_violation ** 2
                                + collision_weight * (clearance <= 0.0), axis=1)
    return (dt * np.sum(squared_distance + 0.02 * effort, axis=1)
            + 10.0 * squared_distance[:, -1] + obstacle_cost)


The cost adds squared position errors along the predicted path, a small control-effort penalty,
and a terminal position penalty. Lower is better. The terminal term encourages progress beyond the next step.
This finite random search is approximate: it does not guarantee an optimal plan or convergence.


For each obstacle, subtract its radius and the robot radius from the center distance.
Negative clearance means collision. Entering the safety margin adds a quadratic penalty;
collision adds a much larger penalty. The obstacle weights and per-step penalty match
`predictive_sampling.py`; the simple goal and effort costs remain those of this tutorial.
These are soft costs evaluated at predicted states, not hard collision constraints.


## 2. Represent a control plan

Sample one independent `(v, omega)` pair per prediction step: **40 scalar parameters**. Each control is held constant over its time step. Adjacent controls can differ sharply.


## 3. Sample, select elites and fit (CEM)

This is the main change from notebook 02. Keep the same `rollout` and `trajectory_cost` functions.
Flatten each `(horizon, 2)` plan into a vector of length `2 * horizon` to estimate elite statistics:

- Sort costs and keep `num_elites` plans with the lowest cost.
- Use `elite_plans.mean(axis=0)` as the next sampling mean.
- Use `np.cov(elite_plans, rowvar=False, bias=True)` as the covariance (normalization by the number of elites).
- Add a small diagonal term `min_std ** 2` so sampling can continue even when elites are similar.

To sample, draw independent uniform noise in `[-1, 1]`, just like notebook 02.
`L = np.linalg.cholesky(covariance)` gives a matrix with `L @ L.T = covariance`.
Multiplying the noise by `sqrt(3) * L.T` gives perturbations with the fitted covariance,
because each original noise component has variance `1/3`.
Before clipping, the resulting distribution is uniform over a transformed box;
its components can be correlated.
This is a CEM-style elite update based on moments, rather than a maximum-likelihood fit of uniform bounds.

Clip sampled controls to the velocity limits before evaluating and fitting them.
Candidate zero preserves the best evaluated plan, so refinement cannot lose it.
The fitted mean guides sampling; the robot executes the best evaluated plan.

At the start of each control step, shift the best plan and the covariance by one time step.
For the new tail, repeat the last control and initialize its covariance with `initial_noise ** 2 / 3`,
with zero correlation to earlier controls. **Do not shift inside the CEM loop.**
Every `covariance_reset_steps` executed steps, reset the entire covariance before sampling;
keep the warm-start plan. `None` disables these periodic resets; `1` resets at every control step.


In [ ]:
def sample_plans(nominal, covariance, best_plan):
    """Draw uniform perturbations shaped by the elite covariance."""
    if nominal is None:
        return rng.uniform(control_low, control_high, (num_samples, horizon, 2))

    # Same uniform noise as notebook 02; now its scale/correlations adapt.
    noise = rng.uniform(-1.0, 1.0, (num_samples, 2 * horizon))
    L = np.linalg.cholesky(covariance)
    perturbations = np.sqrt(3.0) * (noise @ L.T)
    candidates = np.clip(nominal + perturbations.reshape(num_samples, horizon, 2),
                         control_low, control_high)
    candidates[0] = best_plan  # Always retain the best evaluated plan.
    return candidates


In [ ]:
def predictive_sampling(state, previous_plan=None, covariance=None, step=0):
    """Run CEM at a fixed state; also return covariance for the next step."""
    if previous_plan is None:
        nominal = None
        covariance = initial_covariance.copy()
    else:
        # Same warm start as notebook 02: shift only once per control step.
        nominal = np.concatenate((previous_plan[1:], previous_plan[-1:]), axis=0)
        shifted_covariance = initial_covariance.copy()
        if covariance is not None:
            shifted_covariance[:-2, :-2] = covariance[2:, 2:]
        covariance = shifted_covariance

    if covariance_reset_steps is not None and step % covariance_reset_steps == 0:
        covariance = initial_covariance.copy()

    best_plan = nominal
    for iteration in range(cem_iterations):
        plans = sample_plans(nominal, covariance, best_plan)
        controls = plans
        predictions = rollout(state, controls)
        costs = trajectory_cost(predictions, controls)
        best = np.argmin(costs)
        best_plan = plans[best].copy()

        # NEW: fit the next sampling distribution using only the elites.
        elite_indices = np.argsort(costs)[:num_elites]
        elite_plans = plans[elite_indices].reshape(num_elites, 2 * horizon)
        nominal = elite_plans.mean(axis=0).reshape(horizon, 2)
        covariance = np.cov(elite_plans, rowvar=False, bias=True) + covariance_floor

    return best_plan, controls[best].copy(), predictions, best, covariance


## 4. Run the closed loop

Only the first control of the winning plan is executed. The remaining controls are predictions, not a committed path. The stopping condition depends only on position. We explicitly apply zero velocity once inside the tolerance.


In [ ]:
# Reset here so this entire experiment can be repeated independently.
rng = np.random.default_rng(7)
state = start.copy()
previous_plan = None
covariance = initial_covariance.copy()
state_history = [state.copy()]
control_history = []

for step in range(max_steps):
    if np.linalg.norm(state[:2] - goal) <= goal_tolerance:
        break
    best_plan, best_controls, predictions, best, covariance = predictive_sampling(
        state, previous_plan, covariance, step)
    control = best_controls[0]
    state = robot_step(state, control)
    state_history.append(state.copy())
    control_history.append(control.copy())
    previous_plan = best_plan

reached = np.linalg.norm(state[:2] - goal) <= goal_tolerance
if reached:
    control_history.append(np.zeros(2))
    state_history.append(robot_step(state, np.zeros(2)))
state_history = np.asarray(state_history)
control_history = np.asarray(control_history)
print(f"Goal reached: {reached}")
print(f"Final position error: {np.linalg.norm(state[:2] - goal):.3f} m")
print(f"Executed control steps (including stop if reached): {len(control_history)}")

# Check clearance along each executed straight Euler step, not just its endpoints.
segment_start = state_history[:-1, :2]
segment_delta = np.diff(state_history[:, :2], axis=0)
segment_length_squared = np.sum(segment_delta ** 2, axis=1)
minimum_clearance = np.inf
for obstacle_x, obstacle_y, radius in obstacles:
    center = np.array([obstacle_x, obstacle_y])
    projection = np.sum((center - segment_start) * segment_delta, axis=1)
    fraction = np.clip(projection / np.maximum(segment_length_squared, 1e-12), 0, 1)
    closest_point = segment_start + fraction[:, None] * segment_delta
    clearance = np.linalg.norm(closest_point - center, axis=1) - radius - robot_radius
    minimum_clearance = min(minimum_clearance, clearance.min())
print(f"Minimum executed clearance: {minimum_clearance:.3f} m")
print(f"Collision-free executed path: {minimum_clearance > 0}")


## 5. Replay the executed motion

After the experiment finishes, the animation below replays the recorded robot motion.
The blue disk shows the robot footprint, the black line its heading, and the blue trail
its motion so far. Obstacles and safety boundaries remain visible.
Use the playback controls to play, pause, move through individual frames, or replay.
At the default playback speed, each frame corresponds to one control interval `dt`.


In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(state_history[:, 0], state_history[:, 1],
        color="tab:blue", alpha=0.25, label="Executed path")
ax.scatter(*start[:2], label="Start", color="black")
ax.scatter(*goal, marker="*", s=150, label="Goal", color="tab:green")
for index, (obstacle_x, obstacle_y, radius) in enumerate(obstacles):
    ax.add_patch(plt.Circle((obstacle_x, obstacle_y), radius,
                           color="tab:red", alpha=0.6,
                           label="Obstacle" if index == 0 else None))
    ax.add_patch(plt.Circle((obstacle_x, obstacle_y),
                           radius + robot_radius + safety_margin,
                           fill=False, linestyle="--", color="tab:red",
                           label="Robot center safety boundary" if index == 0 else None))

robot = plt.Circle(state_history[0, :2], robot_radius,
                   color="tab:blue", alpha=0.8, label="Robot", zorder=4)
ax.add_patch(robot)
trail, = ax.plot([], [], color="tab:blue", linewidth=2)
heading, = ax.plot([], [], color="black", linewidth=2, zorder=5)
time_label = ax.text(0.02, 0.97, "", transform=ax.transAxes, va="top")
ax.set(xlabel="x [m]", ylabel="y [m]", title="Closed-loop motion")
ax.set_aspect("equal", adjustable="box")
ax.margins(0.1)
ax.autoscale_view()
ax.set_autoscale_on(False)
ax.grid(True)
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1))
fig.tight_layout()


def update_motion(frame):
    x, y, theta = state_history[frame]
    robot.center = (x, y)
    trail.set_data(state_history[:frame + 1, 0], state_history[:frame + 1, 1])
    heading.set_data([x, x + robot_radius * np.cos(theta)],
                     [y, y + robot_radius * np.sin(theta)])
    time_label.set_text(f"Time: {frame * dt:.1f} s")
    return robot, trail, heading, time_label


# Embed playback controls so the recorded experiment can be replayed offline.
motion_animation = FuncAnimation(fig, update_motion, frames=len(state_history),
                                 interval=dt * 1000, repeat=False)
plt.close(fig)
display(HTML(motion_animation.to_jshtml(default_mode="once")))


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
control_time = np.arange(len(control_history)) * dt
for channel, label in enumerate(["v [m/s]", "omega [rad/s]"]):
    axes[channel].step(control_time, control_history[:, channel], where="post")
    axes[channel].set_ylabel(label)
    axes[channel].grid(True)
axes[0].set_title("Executed controls after replanning")
axes[1].set_xlabel("Time [s]")
plt.tight_layout()
plt.show()


## 6. Small experiments

Starting from notebook 02, the implementation requires only three changes:
1. Add the CEM settings and adapt the uniform perturbations using the elite covariance.
2. Wrap sampling and evaluation in a loop; select elites and fit their mean/covariance.
3. Carry covariance through the closed loop, shift it with the plan, and reset it periodically.

Experiments:
- Set `cem_iterations = 1`, then try 3 or 5. The robot still executes one control per step.
- Try `num_elites = 8`, 32 and 128. How does selection affect exploration?
- Compare `covariance_reset_steps = None`, 1, 5 and 20. Does resetting help escape a poor plan?
- Change `initial_noise` (uniform half-widths after resets) and `min_std` (persistent exploration).
- Change the seed and rerun: a single run is not a reliable performance comparison.

CEM evaluates `num_samples * cem_iterations` plans per control step.
For a fair comparison with notebook 02, account for this extra computation and use several seeds.
Even with CEM, this finite search does not guarantee an optimal or collision-free path.
